In [0]:
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'

In [0]:
from pyspark.sql.functions import col

def validate_delta(path, required_columns, pk):

    print(f"Validando: {path}")

    # Verifica se a pasta existe
    try:
        dbutils.fs.ls(path)
    except:
        raise Exception(f"Pasta não encontrada: {path}")

    # Lê o Delta
    df = spark.read.format("delta").load(path)

    # Quantidade de registros
    rows = df.count()

    if rows == 0:
        raise Exception("Tabela vazia.")

    print(f"{rows} registros encontrados.")

    # Colunas obrigatórias

    missing = set(required_columns) - set(df.columns)

    if missing:
        raise Exception(f"Colunas ausentes: {missing}")

    # PK nula

    nulls = df.filter(col(pk).isNull()).count()

    if nulls > 0:
        raise Exception(f"{nulls} registros com {pk} nulo.")

    print("Validação concluída.\n")

In [0]:
# Brands
validate_delta(
    bronze_path + "brand",
    ["brand_id", "brand_name"],
    "brand_id"
)

# Categories
validate_delta(
    bronze_path + "categories",
    ["category_id", "category_name"],
    "category_id"
)

# Customers
validate_delta(
    bronze_path + "customers",
    ["customer_id", "first_name", "last_name","phone", "email"],
    "customer_id"
)

# Orders_items
validate_delta(
    bronze_path + "orders_item",
    ["order_id", "item_id", "product_id", "quantity", "list_price"],
    "order_id"
)

# Orders
validate_delta(
    bronze_path + "orders",
    ["order_id", "customer_id", "order_status", "order_date", "required_date", "shipped_date", "store_id"],
    "order_id"
)

# Products
validate_delta(
    bronze_path + "products",
    ["product_id", "product_name", "brand_id", "category_id", "model_year", "list_price"],
    "product_id")

# staffs
validate_delta(
    bronze_path + "staffs",
    ["staff_id", "first_name", "last_name", "email", "phone"],
    "staff_id")

# stocks
validate_delta(
    bronze_path + "stocks",
    ["store_id", "product_id", "quantity"],
    "product_id")

# stores
validate_delta(
    bronze_path + "stores",
    ["store_id", "store_name", "phone", "email"],
    "store_id")


